In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggl/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# 1. Uninstall the existing CPU-only version
!pip uninstall -y llama-cpp-python

# 2. Install the CUDA-enabled version for Python 3.12
# We use the official 2026 wheel index for CUDA 12.x
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
from llama_cpp import Llama

MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/gguf/14b-gguf/1/Qwen3-14B-Q5_0.gguf"

llm = Llama(
    model_path=MODEL_PATH,
    # 1. Force a high number of layers (Qwen3-14B has 40 blocks, so 41+ ensures all go to GPU)
    n_gpu_layers=99, 
    
    # 2. Split the workload between your two T4s
    tensor_split=[1, 1], 
    
    # 3. Disable mmap to prevent the "CUDA_Host" allocation error
    n_mmap=False, 
    
    # 4. Increase the context now that you have 32GB total VRAM
    n_ctx=8192, 
    
    # 5. Faster processing flags for 2026 architectures
    n_batch=1024,
    verbose=True # Keep this True until you see "all layers assigned to CUDA"
)

In [ ]:
!pip install -q fastapi uvicorn pydantic nest_asyncio pyngrok

In [ ]:
!pip install -q fpdf2

In [ ]:
!npm install -g localtunnel

In [2]:
import os
import signal
import threading
import uvicorn
import nest_asyncio
import time
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional
from llama_cpp import Llama

# --- 0. FORCE CLEANUP ---
# Kills anything on port 8000 to prevent [Errno 98]
try:
    print("🧹 Clearing port 8000...")
    !fuser -k 8000/tcp
    time.sleep(2) 
except:
    pass

# --- 1. DEFINE DATA MODELS (Fixes NameError) ---
class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = 1024

# --- 2. LOAD MODEL (Dual T4 Optimization) ---
MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/gguf/14b-gguf/1/Qwen3-14B-Q5_0.gguf"

if 'llm' not in globals():
    print("🚀 Loading Qwen3 onto Dual T4 GPUs (32GB VRAM)...")
    llm = Llama(
        model_path=MODEL_PATH,
        n_gpu_layers=-1,      # All layers to GPU
        tensor_split=[1, 1],  # 50/50 split across both T4s
        n_ctx=8192,           # High context for coding
        n_batch=1024,
        flash_attn=True,      # Much faster on T4s
        verbose=False
    )
else:
    print("✅ Model already loaded in VRAM.")

# --- 3. FASTAPI SETUP WITH CORS ---
app = FastAPI(title="Qwen3 Dual-T4 API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],      # CRITICAL: Allows your .html file to connect
    allow_credentials=True,
    allow_methods=["*"],      # Allows POST, OPTIONS, etc.
    allow_headers=["*"],
)

nest_asyncio.apply()

@app.post("/v1/chat/completions")
async def chat_endpoint(request: ChatRequest):
    try:
        # Convert Pydantic objects to simple dictionaries
        formatted_messages = [msg.model_dump() for msg in request.messages]
        
        # Call the dual-GPU model
        response = llm.create_chat_completion(
            messages=formatted_messages,
            temperature=request.temperature,
            max_tokens=request.max_tokens
        )
        return response
    except Exception as e:
        print(f"❌ API Error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

# --- 4. START SERVER IN BACKGROUND ---
def run_api():
    print("🌐 Starting FastAPI Server on port 8000...")
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Check if thread is already running to avoid duplicates
api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print("\n✨ SYSTEM READY! ✨")
print("1. Ensure Localtunnel is running in another cell: !lt --port 8000")
print("2. Use the URL in your .html app or Windows CMD.")

🧹 Clearing port 8000...
✅ Model already loaded in VRAM.
🌐 Starting FastAPI Server on port 8000...

✨ SYSTEM READY! ✨
1. Ensure Localtunnel is running in another cell: !lt --port 8000
2. Use the URL in your .html app or Windows CMD.


INFO:     Started server process [1781]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


# 4. Local Notebook Chat Loop
print("\n--- NOTEBOOK CHAT ACTIVE ---")
history = [{"role": "system", "content": "You are a coding assistant."}]

while True:
    user_input = input("\nUser: ")
    if user_input.lower() in ['exit', 'quit']: break
    if user_input.lower() == 'clear': 
        history = [history[0]]
        print("Chat cleared.")
        continue

    history.append({"role": "user", "content": user_input})
    print("\nAssistant: ", end="", flush=True)
    
    full_resp = ""
    for chunk in llm.create_chat_completion(messages=history, stream=True):
        token = chunk["choices"][0].get("delta", {}).get("content", "")
        if token:
            print(token, end="", flush=True)
            full_resp += token
    
    history.append({"role": "assistant", "content": full_resp})
    print()

LOCAL CHECK chnage to code when need 

In [ ]:
from pyngrok import ngrok

# Replace 'YOUR_NGROK_AUTHTOKEN' with your free token from ngrok.com
# !ngrok config add-authtoken YOUR_NGROK_AUTHTOKEN

public_url = ngrok.connect(8000)
print(f"🌍 Your Public API URL: {public_url}")

NGROK

In [3]:
# Run this to get your password for the tunnel
print("🔑 Your Endpoint Password (Kaggle IP):")
!curl ipv4.icanhazip.com

🔑 Your Endpoint Password (Kaggle IP):
34.133.197.104


In [4]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from fpdf import FPDF
import datetime
import uuid

# --- JAVASCRIPT INJECTION ---
# This script handles the clipboard logic and UI feedback
JS_COPY_SCRIPT = """
<script>
function copyToClipboard(button, codeId) {
    const code = document.getElementById(codeId).innerText;
    navigator.clipboard.writeText(code).then(() => {
        const originalText = button.innerText;
        button.innerText = "✅ Copied!";
        button.classList.add("copied");
        setTimeout(() => {
            button.innerText = originalText;
            button.classList.remove("copied");
        }, 2000);
    });
}
</script>
<style>
    .user-msg { background: #2d2d2d; color: white; padding: 12px; border-radius: 12px; margin: 8px 0; border-left: 5px solid #007bff; box-shadow: 2px 2px 5px rgba(0,0,0,0.2); }
    .bot-msg { background: #1e1e1e; color: #ddd; padding: 12px; border-radius: 12px; margin: 8px 0; border-left: 5px solid #28a745; box-shadow: 2px 2px 5px rgba(0,0,0,0.2); }
    .thinking { color: #888; font-style: italic; font-size: 0.9em; background: #252525; padding: 8px; border-radius: 5px; margin-bottom: 10px; border-bottom: 1px dashed #444; }
    .code-container { background: #000; border-radius: 6px; margin: 12px 0; border: 1px solid #333; overflow: hidden; }
    .code-header { background: #333; padding: 5px 10px; display: flex; justify-content: space-between; align-items: center; color: #ccc; font-size: 0.8em; }
    .copy-btn { background: #444; color: white; border: none; padding: 2px 8px; border-radius: 4px; cursor: pointer; transition: 0.2s; }
    .copy-btn:hover { background: #666; }
    .copy-btn.copied { background: #28a745; }
    .code-block { color: #00ff00; font-family: 'Consolas', 'Monaco', monospace; padding: 15px; overflow-x: auto; white-space: pre; margin: 0; }
</style>
"""

# --- UI STATE ---
chat_history = []
is_running = True

def format_content(text):
    # Handle Thinking Block
    if "<think>" in text:
        text = text.replace("<think>", "<div class='thinking'>💭 Thinking...<br>").replace("</think>", "</div>")
    
    # Code Block Parsing with Unique IDs for JS
    parts = text.split("```")
    formatted = ""
    for i, part in enumerate(parts):
        if i % 2 == 1: # Inside a code block
            code_id = f"code-{uuid.uuid4().hex[:8]}"
            # Split language if exists
            lines = part.split('\n', 1)
            lang = lines[0].strip() if len(lines) > 1 else "code"
            code_content = lines[1] if len(lines) > 1 else lines[0]
            
            formatted += f'''
            <div class="code-container">
                <div class="code-header">
                    <span>{lang}</span>
                    <button class="copy-btn" onclick="copyToClipboard(this, '{code_id}')">📋 Copy</button>
                </div>
                <pre id="{code_id}" class="code-block">{code_content.strip()}</pre>
            </div>
            '''
        else:
            formatted += part.replace("\n", "<br>")
    return formatted

# --- WIDGETS ---
output_area = widgets.Output(layout={'border': '1px solid #444', 'height': '450px', 'overflow_y': 'scroll'})
input_box = widgets.Textarea(placeholder='Ask for code, logic, or debugging...', layout={'width': '100%', 'height': '100px'})
send_btn = widgets.Button(description='Send', button_style='primary', icon='paper-plane')
stop_btn = widgets.Button(description='Stop', button_style='danger', icon='stop')
clear_btn = widgets.Button(description='Clear Chat', icon='trash')
pdf_btn = widgets.Button(description='Download PDF', button_style='success', icon='file-pdf-o')

def render_chat():
    with output_area:
        clear_output()
        display(HTML(JS_COPY_SCRIPT))
        for msg in chat_history:
            if msg['role'] == "system": continue
            cls = "user-msg" if msg['role'] == "user" else "bot-msg"
            display(HTML(f"<div class='{cls}'><b>{msg['role'].upper()}:</b><br>{format_content(msg['content'])}</div>"))

# --- PDF LOGIC ---
def on_pdf(b):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", 'B', 16)
    pdf.cell(0, 10, "Chat Export", ln=True, align='C')
    pdf.ln(10)
    pdf.set_font("Helvetica", size=10)
    
    for msg in chat_history:
        pdf.set_font("Helvetica", 'B', 10)
        pdf.cell(0, 10, f"{msg['role'].upper()}:", ln=True)
        pdf.set_font("Helvetica", size=10)
        pdf.multi_cell(0, 5, txt=msg['content'].encode('latin-1', 'replace').decode('latin-1'))
        pdf.ln(5)
        
    path = f"qwen_chat_{datetime.datetime.now().strftime('%H%M%S')}.pdf"
    pdf.output(path)
    print(f"✅ PDF exported to /working/{path}")

# --- HANDLERS ---
def on_send(b):
    if not is_running or not input_box.value.strip(): return
    
    user_text = input_box.value
    input_box.value = ""
    chat_history.append({"role": "user", "content": user_text})
    render_chat()
    
    full_resp = ""
    # Streaming from the existing 'llm' instance
    stream = llm.create_chat_completion(messages=chat_history, stream=True)
    
    for chunk in stream:
        if not is_running: break
        token = chunk["choices"][0].get("delta", {}).get("content", "")
        if token:
            full_resp += token
            
    chat_history.append({"role": "assistant", "content": full_resp})
    render_chat()

def on_stop(b):
    global is_running
    is_running = not is_running
    stop_btn.description = "Start" if not is_running else "Stop"
    stop_btn.button_style = "success" if not is_running else "danger"

send_btn.on_click(on_send)
stop_btn.on_click(on_stop)
pdf_btn.on_click(on_pdf)
clear_btn.on_click(lambda x: (chat_history.clear(), chat_history.append({"role": "system", "content": "Assistant"}), render_chat()))

# --- DISPLAY ---
chat_history.append({"role": "system", "content": "Assistant"})
display(widgets.VBox([
    widgets.HTML("<h2 style='color: #28a745;'>Qwen3 Developer Studio (2x T4)</h2>"),
    output_area,
    input_box,
    widgets.HBox([send_btn, stop_btn, clear_btn, pdf_btn])
]))

In [ ]:
# This will generate your public URL
!lt --port 8000

your url is: https://social-poets-flash.loca.lt
INFO:     81.17.122.62:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     81.17.122.62:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     81.17.122.62:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     162.245.86.180:0 - "HEAD / HTTP/1.1" 404 Not Found
INFO:     81.17.122.62:0 - "OPTIONS /%20/v1/chat/completions HTTP/1.1" 200 OK
INFO:     81.17.122.62:0 - "POST /%20/v1/chat/completions HTTP/1.1" 404 Not Found
INFO:     81.17.122.62:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     81.17.122.62:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     81.17.122.62:0 - "OPTIONS /v1/chat/completions HTTP/1.1" 200 OK
INFO:     81.17.122.62:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     81.17.122.62:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     81.17.122.62:0 - "OPTIONS /v1/chat/completions HTTP/1.1" 200 OK
INFO:     81.17.122.62:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     81.17.122.62:0 - "OPTIONS /v1/chat/completions HTTP/1.1" 20

curl https://nasty-worlds-find.loca.lt/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "messages": [{"role": "user", "content": "Hello Qwen!"}],
    "temperature": 0.7
  }'

https://nasty-worlds-find.loca.lt/docs

@app.post("/v1/chat/completions")